DATASET 2 — ONLINE RETAIL II (VALIDATED)

COMMON IMPORTS (USE IN ALL NOTEBOOKS)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import warnings
warnings.filterwarnings('ignore')

1. Data Loading

In [ ]:
# Note: Online Retail II has two sheets
sales_2009 = pd.read_excel(
    "../data/raw/online_retail.xlsx",
    sheet_name='Year 2009-2010'
)

sales_2010 = pd.read_excel(
    "../data/raw/online_retail.xlsx",
    sheet_name='Year 2010-2011'
)

# Combine both years
sales = pd.concat([sales_2009, sales_2010], ignore_index=True)

print(f"📦 Combined dataset shape: {sales.shape}")
print(f"📋 Columns: {sales.columns.tolist()}")

2. Data Cleaning

In [ ]:
print("\n🔍 Initial data quality:")
print(sales.isnull().sum())

# Remove rows without Customer ID (essential for analysis)
sales = sales[sales['Customer ID'].notna()]

# Remove cancelled transactions (Invoice starts with 'C')
sales = sales[~sales['Invoice'].astype(str).str.startswith('C')]

# Remove invalid quantities and prices
sales = sales[sales['Quantity'] > 0]
sales = sales[sales['Price'] > 0]

# Remove extreme outliers (likely data errors)
sales = sales[sales['Quantity'] < sales['Quantity'].quantile(0.999)]
sales = sales[sales['Price'] < sales['Price'].quantile(0.999)]

# Calculate revenue
sales['Revenue'] = sales['Quantity'] * sales['Price']

# Parse dates
sales['InvoiceDate'] = pd.to_datetime(sales['InvoiceDate'])
sales['month'] = sales['InvoiceDate'].dt.to_period('M')
sales['year'] = sales['InvoiceDate'].dt.year

# Keep only products with ≥6 months of sales history
product_months = sales.groupby('StockCode')['month'].nunique()
valid_products = product_months[product_months >= 6].index
sales = sales[sales['StockCode'].isin(valid_products)]

print(f"\n✅ After cleaning: {sales.shape[0]} transactions")
print(f"✅ Products analyzed: {sales['StockCode'].nunique()}")
print(f"✅ Date range: {sales['InvoiceDate'].min()} to {sales['InvoiceDate'].max()}")


3. Feature Engineering

In [ ]:
## Monthly aggregation per product
monthly_sales = (
    sales
    .groupby(['StockCode', 'month'])
    .agg({
        'Revenue': ['sum', 'mean', 'std', 'count'],
        'Quantity': 'sum',
        'Customer ID': 'nunique',
        'Invoice': 'nunique',
        'Price': 'mean'
    })
    .reset_index()
)

monthly_sales.columns = [
    'StockCode', 'month',
    'revenue_total', 'revenue_mean', 'revenue_std', 'transaction_count',
    'quantity_sold', 'unique_customers', 'unique_orders',
    'avg_price'
]

# Fill NaN std with 0 (single transaction months)
monthly_sales['revenue_std'] = monthly_sales['revenue_std'].fillna(0)

## Product lifecycle metrics
first_sale = (
    sales
    .groupby('StockCode')['InvoiceDate']
    .min()
    .reset_index()
    .rename(columns={'InvoiceDate': 'first_sale_date'})
)

monthly_sales = monthly_sales.merge(first_sale, on='StockCode')
monthly_sales['month_date'] = monthly_sales['month'].dt.to_timestamp()
monthly_sales['product_age_months'] = (
    (monthly_sales['month_date'] - monthly_sales['first_sale_date']).dt.days / 30
).round(1)

## Lifecycle stage
def classify_lifecycle(age_months):
    if age_months < 3:
        return 'introduction'
    elif age_months < 12:
        return 'growth'
    elif age_months < 24:
        return 'maturity'
    else:
        return 'decline'

monthly_sales['lifecycle_stage'] = (
    monthly_sales['product_age_months'].apply(classify_lifecycle)
)

## FATIGUE SIGNALS

# 1. Revenue velocity (% change month-over-month)
monthly_sales['revenue_velocity'] = (
    monthly_sales
    .groupby('StockCode')['revenue_total']
    .transform(lambda x: x.pct_change() * 100)
)

# 2. Revenue acceleration
monthly_sales['revenue_acceleration'] = (
    monthly_sales
    .groupby('StockCode')['revenue_velocity']
    .transform(lambda x: x.diff())
)

# 3. Customer churn rate
monthly_sales['customer_churn_rate'] = (
    monthly_sales
    .groupby('StockCode')['unique_customers']
    .transform(lambda x: x.pct_change() * 100)
)

# 4. Revenue volatility (instability)
monthly_sales['revenue_volatility'] = (
    monthly_sales
    .groupby('StockCode')['revenue_total']
    .transform(lambda x: x.rolling(3, min_periods=1).std())
)

# 5. Order frequency decline
monthly_sales['order_frequency_change'] = (
    monthly_sales
    .groupby('StockCode')['unique_orders']
    .transform(lambda x: x.pct_change() * 100)
)

# 6. Average order value change
monthly_sales['aov'] = monthly_sales['revenue_total'] / monthly_sales['unique_orders']
monthly_sales['aov_change'] = (
    monthly_sales
    .groupby('StockCode')['aov']
    .transform(lambda x: x.pct_change() * 100)
)

# 7. Market concentration (customer diversity)
monthly_sales['customer_concentration'] = (
    monthly_sales['unique_customers'] / monthly_sales['transaction_count']
)


4. EDA — Emotional Fatigue

In [ ]:
## 🔹 UNIVARIATE ANALYSIS

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# Revenue distribution
sns.histplot(
    data=monthly_sales,
    x='revenue_total',
    bins=50,
    kde=True,
    log_scale=True,
    ax=axes[0, 0]
)
axes[0, 0].set_title('Revenue Distribution (Log Scale)', fontsize=12, fontweight='bold')

# Revenue by lifecycle
sns.boxplot(
    data=monthly_sales,
    x='lifecycle_stage',
    y='revenue_total',
    order=['introduction', 'growth', 'maturity', 'decline'],
    ax=axes[0, 1]
)
axes[0, 1].set_title('Revenue by Lifecycle Stage', fontsize=12, fontweight='bold')
axes[0, 1].set_yscale('log')

# Revenue velocity
sns.histplot(
    data=monthly_sales[monthly_sales['revenue_velocity'].notna()],
    x='revenue_velocity',
    bins=60,
    kde=True,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Revenue Velocity (Monthly % Change)', fontsize=12, fontweight='bold')
axes[1, 0].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_xlim(-100, 100)

# Customer churn
sns.histplot(
    data=monthly_sales[monthly_sales['customer_churn_rate'].notna()],
    x='customer_churn_rate',
    bins=60,
    kde=True,
    ax=axes[1, 1]
)
axes[1, 1].set_title('Customer Churn Rate (%)', fontsize=12, fontweight='bold')
axes[1, 1].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_xlim(-100, 100)

# Revenue volatility
sns.histplot(
    data=monthly_sales,
    x='revenue_volatility',
    bins=50,
    kde=True,
    ax=axes[2, 0]
)
axes[2, 0].set_title('Revenue Volatility (Instability)', fontsize=12, fontweight='bold')

# Customer concentration
sns.histplot(
    data=monthly_sales,
    x='customer_concentration',
    bins=50,
    kde=True,
    ax=axes[2, 1]
)
axes[2, 1].set_title('Customer Concentration (Diversity)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/sales_univariate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 🔹 BIVARIATE ANALYSIS

# Identify declining products
declining_products = (
    monthly_sales[
        (monthly_sales['revenue_velocity'] < -15) &
        (monthly_sales['product_age_months'] >= 6)
    ]
    .groupby('StockCode')
    .size()
    .nlargest(10)
    .index
)

decline_data = monthly_sales[
    monthly_sales['StockCode'].isin(declining_products)
].copy()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Revenue trajectory
for product in declining_products:
    product_data = decline_data[decline_data['StockCode'] == product]
    axes[0].plot(
        product_data['product_age_months'],
        product_data['revenue_total'],
        marker='o',
        alpha=0.7,
        linewidth=2
    )

axes[0].set_xlabel('Product Age (Months)', fontsize=11)
axes[0].set_ylabel('Monthly Revenue', fontsize=11)
axes[0].set_title('Financial Fatigue Trajectories (Top 10 Declining Products)',
                  fontsize=12, fontweight='bold')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Customer trajectory
for product in declining_products:
    product_data = decline_data[decline_data['StockCode'] == product]
    axes[1].plot(
        product_data['product_age_months'],
        product_data['unique_customers'],
        marker='s',
        alpha=0.7,
        linewidth=2
    )

axes[1].set_xlabel('Product Age (Months)', fontsize=11)
axes[1].set_ylabel('Unique Customers', fontsize=11)
axes[1].set_title('Customer Base Erosion (Same Products)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/sales_fatigue_trajectories.png', dpi=300, bbox_inches='tight')
plt.show()

## 🔹 CORRELATION HEATMAP

corr_features = [
    'revenue_total', 'revenue_velocity', 'revenue_acceleration',
    'revenue_volatility', 'customer_churn_rate', 'order_frequency_change',
    'aov_change', 'product_age_months'
]

corr_matrix = monthly_sales[corr_features].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=1,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title('Financial Fatigue Signal Correlation Heatmap',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/sales_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 🔹 OUTLIER DETECTION

monthly_sales['z_revenue_velocity'] = zscore(
    monthly_sales['revenue_velocity'].fillna(0)
)
monthly_sales['z_customer_churn'] = zscore(
    monthly_sales['customer_churn_rate'].fillna(0)
)

outliers = monthly_sales[
    (monthly_sales['z_revenue_velocity'] < -3) |
    (monthly_sales['z_customer_churn'] < -3)
]

print(f"\n⚠️ Extreme financial fatigue outliers: {len(outliers)} product-months")
print(f"   Representing {outliers['StockCode'].nunique()} unique products")


5. FATIGUE LABELING (MULTI-CRITERIA)

In [ ]:
def label_financial_fatigue(row):
    """
    Multi-dimensional financial fatigue classification.
    """
    
    velocity = row['revenue_velocity'] if pd.notna(row['revenue_velocity']) else 0
    acceleration = row['revenue_acceleration'] if pd.notna(row['revenue_acceleration']) else 0
    churn = row['customer_churn_rate'] if pd.notna(row['customer_churn_rate']) else 0
    volatility = row['revenue_volatility'] if pd.notna(row['revenue_volatility']) else 0
    
    # High fatigue
    if (
        velocity < -20 and
        churn < -15 and
        acceleration < -10
    ):
        return 'high_fatigue'
    
    # Moderate fatigue
    elif (
        velocity < -10 or
        (churn < -10 and volatility > row['revenue_total'] * 0.5)
    ):
        return 'moderate_fatigue'
    
    # Healthy
    else:
        return 'healthy'

monthly_sales['fatigue_label'] = monthly_sales.apply(
    label_financial_fatigue, axis=1
)

print("\n📊 FINANCIAL FATIGUE DISTRIBUTION:")
print(monthly_sales['fatigue_label'].value_counts())


6. EXPORT PROCESSED DATA

In [ ]:
monthly_sales.to_csv(
    '../data/processed/sales_fatigue_signals.csv',
    index=False
)

print("\n✅ SALES EDA COMPLETE!")
print(f"📁 Saved: data/processed/sales_fatigue_signals.csv")